# Chapter 11: Calculating with Conics

**Source span.** Perspectives on Projective Geometry, Chapter 11, Sections 11.1-11.6, printed pages 189-208 / PDF pages 211-230.

**Chapter goal.** Turn conics from static quadratic curves into computable projective objects. The same matrix representation will split degenerate conics, explain why branching is unavoidable, reduce conic-line intersection to a square-root split, reduce conic-conic intersection to a pencil cubic plus two line solves, track complex roots, and construct the two conics through four points tangent to a chosen line.

The textbook source is used here for orientation and terminology only. The prose, examples, code, diagrams, and checks below are original teaching material.

## Computational Translation Guide

A point is a homogeneous vector `p = (x, y, z)`. A line is a homogeneous vector `l = (a, b, c)` with incidence `l dot p = 0`. A conic is a symmetric matrix `A`; its points satisfy `p.T @ A @ p = 0`.

The chapter's recipes use four translations. A rank-2 degenerate conic that is the union of two lines `g` and `h` has a matrix proportional to `g h.T + h g.T`. A skew matrix `M_v` does not change a quadratic form, but adding the right one can turn that symmetric matrix into a rank-1 outer product. For a line `l`, the dual degenerate conic `M_l.T @ A @ M_l` splits into the two conic-line intersection points. For two conics `A` and `B`, the degenerate members of the pencil `lambda A + mu B` are found from the homogeneous cubic `det(lambda A + mu B) = 0`.

Library routing: `NumPy` handles homogeneous linear algebra, `SymPy` verifies the exact skew identity, `Matplotlib` gives durable conic/incidence diagrams, `Plotly` powers the line-conic parameter lab, `NetworkX` displays the decision-tree checks, and `pandas` stores inspection tables.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Run from inside the Perspectives-on-Projective-Geometry book tree.")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import assert_artifacts, book_relative, display_artifact, image_stats, save_json, save_table

CHAPTER_SLUG = "chapter-11-calculating-with-conics"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
for subdir in ["figures", "html", "tables", "checks"]:
    (ARTIFACT_ROOT / subdir).mkdir(parents=True, exist_ok=True)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"

TOL = 1e-9
plt.rcParams.update({"figure.dpi": 130, "font.size": 9})
print(book_relative(ARTIFACT_ROOT))

## Visualization Planner Pass

This planner pass is source-specific. Each visual below is tied to one Chapter 11 operation, an inspection target, and an executable invariant.

In [ ]:
storyboard = {
    "chapter goal": "Compute with conics using homogeneous matrices: split degenerates, branch safely, intersect conics with lines and conics, handle complex roots, and impose one tangent constraint.",
    "source span read": "Printed pp. 189-208 / PDF pp. 211-230, Sections 11.1-11.6.",
    "concept inventory": [
        "rank-2 degenerate conic A = g h^T + h g^T and skew correction to rank one",
        "rank-1 double line and the need to inspect a nonzero matrix entry",
        "line-conic intersection as splitting M_l^T A M_l",
        "conic-conic intersection via the determinant cubic of a pencil",
        "complex conjugate roots and projective real representatives",
        "four points plus one tangent line through fixed points of an involution",
        "decision-tree checks for ranks, pivots, roots, and residuals",
    ],
    "library routing table": [
        {"concept": "homogeneous matrix operations", "representation": "small dense matrices", "library": "NumPy", "why": "direct ranks, null vectors, skew corrections, and residuals"},
        {"concept": "skew split identity", "representation": "exact symbolic matrices", "library": "SymPy", "why": "verifies A + M_{g x h} = 2 g h^T exactly"},
        {"concept": "conic and line incidence", "representation": "labeled planar diagrams", "library": "Matplotlib", "why": "durable static figures with equal aspect and contours"},
        {"concept": "line-conic root parameter", "representation": "slider over line position", "library": "Plotly", "why": "the real/tangent/complex transition is a parameter phenomenon"},
        {"concept": "branching and implementation checks", "representation": "directed graph", "library": "NetworkX", "why": "the algorithms are decision trees"},
        {"concept": "inspection tables", "representation": "CSV summaries", "library": "pandas", "why": "keeps branch triggers and residuals auditable"},
    ],
    "visual sequence": [
        {"artifact": "figures/degenerate-conic-splitting.png", "inspection target": "the symmetric union matrix and the skew-corrected rank-one matrix encode the same quadratic form", "validation": "rank-one score and recovered-line incidence"},
        {"artifact": "figures/branching-double-line-path.png", "inspection target": "a closed path of double-line matrices cannot choose a continuous vector representative without a branch", "validation": "pivot changes and endpoint identification"},
        {"artifact": "figures/conic-line-root-splitting.png", "inspection target": "secant, tangent, and exterior lines differ by the split becoming real, double, or complex", "validation": "line incidence and conic residuals"},
        {"artifact": "html/conic-line-root-discriminant-lab.html", "inspection target": "move the line through an ellipse and watch the discriminant cross zero", "validation": "root residual table"},
        {"artifact": "figures/conic-pencil-degenerate-members.png", "inspection target": "a determinant cubic selects line-pair members of the conic pencil", "validation": "cubic residuals and common-point residuals"},
        {"artifact": "figures/complex-conjugate-reality-check.png", "inspection target": "complex conjugate points can join to a real line", "validation": "imaginary part after projective rescaling"},
        {"artifact": "figures/tangent-conics-four-points-line.png", "inspection target": "two fixed points on the line generate two tangent conics through four points", "validation": "dual tangent residual"},
        {"artifact": "figures/decision-tree-checks.png", "inspection target": "every recipe has a branch trigger and an invariant", "validation": "decision table saved as CSV"},
    ],
    "proof-visualization strategy": "Use an exact skew identity, a branch-forcing double-line path, and an implementation decision graph instead of a decorative route map.",
}
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
pd.DataFrame(storyboard["visual sequence"])

## Local Algebra Toolkit

These helpers are chapter-local and intentionally small. They keep the mathematical inputs and outputs visible: homogeneous points, lines, conic matrices, skew matrices, adjugates, split factors, and residuals.

In [ ]:
def hpoint(x, y, z=1.0):
    return np.array([x, y, z], dtype=complex)


def affine(p):
    p = np.asarray(p, dtype=complex)
    return p[:2] / p[2]


def normalize_h(v, tol=TOL):
    v = np.asarray(v, dtype=complex)
    idx = int(np.argmax(np.abs(v)))
    if abs(v[idx]) < tol:
        return v.copy()
    w = v / v[idx]
    if np.linalg.norm(w.imag) < 1e-8:
        return w.real.astype(float)
    return w


def real_representative(v, tol=1e-8):
    w = normalize_h(v, tol=tol)
    if np.linalg.norm(np.asarray(w).imag) < tol:
        return np.asarray(w).real.astype(float)
    raise ValueError(f"not a real projective representative: {w}")


def join_points(p, q):
    return np.cross(np.asarray(p, dtype=complex), np.asarray(q, dtype=complex))


def meet_lines(g, h):
    return np.cross(np.asarray(g, dtype=complex), np.asarray(h, dtype=complex))


def bracket(a, b, c):
    return np.linalg.det(np.column_stack([a, b, c]))


def skew(v):
    x, y, z = np.asarray(v, dtype=complex)
    return np.array([[0, z, -y], [-z, 0, x], [y, -x, 0]], dtype=complex)


def adjugate(M):
    M = np.asarray(M, dtype=complex)
    return np.array([
        [M[1, 1] * M[2, 2] - M[1, 2] * M[2, 1], M[0, 2] * M[2, 1] - M[0, 1] * M[2, 2], M[0, 1] * M[1, 2] - M[0, 2] * M[1, 1]],
        [M[1, 2] * M[2, 0] - M[1, 0] * M[2, 2], M[0, 0] * M[2, 2] - M[0, 2] * M[2, 0], M[0, 2] * M[1, 0] - M[0, 0] * M[1, 2]],
        [M[1, 0] * M[2, 1] - M[1, 1] * M[2, 0], M[0, 1] * M[2, 0] - M[0, 0] * M[2, 1], M[0, 0] * M[1, 1] - M[0, 1] * M[1, 0]],
    ], dtype=complex)


def line_pair_matrix(g, h):
    return np.outer(g, h) + np.outer(h, g)


def conic_from_five(points):
    rows = []
    for p in points:
        x, y, z = np.asarray(p, dtype=complex)
        rows.append([x * x, 2 * x * y, y * y, 2 * x * z, 2 * y * z, z * z])
    _, _, vh = np.linalg.svd(np.asarray(rows, dtype=complex))
    a, b, c, d, e, f = vh[-1, :]
    M = np.array([[a, b, d], [b, c, e], [d, e, f]], dtype=complex)
    M = M / np.linalg.norm(M)
    return M.real if np.linalg.norm(M.imag) < 1e-10 else M


def conic_value(M, p):
    p = np.asarray(p, dtype=complex)
    return p @ np.asarray(M, dtype=complex) @ p


def matrix_rank_score(M):
    s = np.linalg.svd(np.asarray(M, dtype=complex), compute_uv=False)
    return 0.0 if len(s) < 2 or abs(s[0]) < TOL else float(abs(s[1] / s[0]))


def split_degenerate_conic(A, known_join=None, tol=1e-9):
    A = np.asarray(A, dtype=complex)
    s = np.linalg.svd(A, compute_uv=False)
    rank = int(np.sum(s > tol * max(float(abs(s[0])), 1.0)))
    if rank <= 1:
        i, j = np.unravel_index(np.argmax(np.abs(A)), A.shape)
        return normalize_h(A[:, j]), normalize_h(A[i, :]), {"case": "double", "rank_score": 0.0, "alpha": 0j}
    if known_join is None:
        _, _, vh = np.linalg.svd(A)
        pivot_vector = vh.conj().T[:, -1]
    else:
        pivot_vector = np.asarray(known_join, dtype=complex)
    M = skew(pivot_vector)
    candidates = []
    for i in range(3):
        for j in range(i + 1, 3):
            m = M[i, j]
            if abs(m) > tol:
                root = np.sqrt(complex((A[i, j] ** 2 - A[i, i] * A[j, j]) / (m ** 2)))
                candidates.extend([root, -root])
    if not candidates:
        raise ValueError("could not find a nonzero skew pivot")
    score, alpha, C = min(((matrix_rank_score(A + a * M), a, A + a * M) for a in candidates), key=lambda item: item[0])
    i, j = np.unravel_index(np.argmax(np.abs(C)), C.shape)
    return normalize_h(C[:, j]), normalize_h(C[i, :]), {"case": "pair", "rank_score": float(score), "alpha": complex(alpha)}


def intersect_conic_line(A, l):
    l = np.asarray(l, dtype=complex)
    dual_pair = skew(l).T @ np.asarray(A, dtype=complex) @ skew(l)
    p, q, info = split_degenerate_conic(dual_pair, known_join=l)
    return p, q, {**info, "dual_pair": dual_pair}


def pencil_cubic_coefficients(A, B):
    Acols = [np.asarray(A, dtype=complex)[:, i] for i in range(3)]
    Bcols = [np.asarray(B, dtype=complex)[:, i] for i in range(3)]
    det = lambda cols: np.linalg.det(np.column_stack(cols))
    alpha = det(Acols)
    beta = det([Acols[0], Acols[1], Bcols[2]]) + det([Acols[0], Bcols[1], Acols[2]]) + det([Bcols[0], Acols[1], Acols[2]])
    gamma = det([Acols[0], Bcols[1], Bcols[2]]) + det([Bcols[0], Acols[1], Bcols[2]]) + det([Bcols[0], Bcols[1], Acols[2]])
    delta = det(Bcols)
    coeffs = np.array([alpha, beta, gamma, delta], dtype=complex)
    return coeffs.real if np.linalg.norm(coeffs.imag) < 1e-10 else coeffs


def plot_line(ax, l, xlim, ylim, **kwargs):
    a, b, c = np.asarray(l, dtype=complex).real
    if abs(b) >= abs(a):
        xs = np.linspace(xlim[0], xlim[1], 220)
        ys = -(a * xs + c) / b
        mask = (ys >= ylim[0] - 0.1) & (ys <= ylim[1] + 0.1)
        ax.plot(xs[mask], ys[mask], **kwargs)
    else:
        ys = np.linspace(ylim[0], ylim[1], 220)
        xs = -(b * ys + c) / a
        mask = (xs >= xlim[0] - 0.1) & (xs <= xlim[1] + 0.1)
        ax.plot(xs[mask], ys[mask], **kwargs)


def plot_conic(ax, M, xlim, ylim, color="black", linewidth=1.8, linestyle="-", label=None, alpha=1.0):
    xs = np.linspace(xlim[0], xlim[1], 420)
    ys = np.linspace(ylim[0], ylim[1], 420)
    X, Y = np.meshgrid(xs, ys)
    a, b, c = M[0, 0].real, M[0, 1].real, M[1, 1].real
    d, e, f = M[0, 2].real, M[1, 2].real, M[2, 2].real
    Z = a * X * X + 2 * b * X * Y + c * Y * Y + 2 * d * X + 2 * e * Y + f
    ax.contour(X, Y, Z, levels=[0], colors=[color], linewidths=linewidth, linestyles=linestyle, alpha=alpha)
    if label:
        ax.plot([], [], color=color, linewidth=linewidth, linestyle=linestyle, label=label, alpha=alpha)


def scatter_hpoints(ax, points, **kwargs):
    pts = np.array([affine(p) for p in points], dtype=complex)
    ax.scatter(pts[:, 0].real, pts[:, 1].real, **kwargs)


def save_figure(fig, filename):
    path = FIGURES / filename
    fig.savefig(path, bbox_inches="tight", dpi=180)
    plt.close(fig)
    return path


def cross_ratio(a, b, c, d):
    return ((a - c) * (b - d)) / ((a - d) * (b - c))

## 1. Splitting a Degenerate Conic

A rank-2 degenerate conic is the union of two lines. The symmetric matrix hides the ordered factors because `g h.T` and `h g.T` have been added. The skew matrix restores the missing antisymmetric half: it does not change the quadratic form, but it can turn the matrix into a rank-1 outer product whose columns and rows reveal the two lines.

In [ ]:
g_sym = sp.Matrix([2, -1, 3])
h_sym = sp.Matrix([1, 4, -2])
p_sym = g_sym.cross(h_sym)
M_sym = sp.Matrix([[0, p_sym[2], -p_sym[1]], [-p_sym[2], 0, p_sym[0]], [p_sym[1], -p_sym[0], 0]])
A_sym = g_sym * h_sym.T + h_sym * g_sym.T
sympy_identity_ok = (A_sym + M_sym) == (2 * g_sym * h_sym.T)
assert sympy_identity_ok
sympy_identity_ok

In [ ]:
g = np.array([-0.55, 1.0, -0.25])
h = np.array([0.80, 1.0, 0.35])
A_degenerate = line_pair_matrix(g, h)
intersection = meet_lines(g, h)
recovered_g, recovered_h, split_info = split_degenerate_conic(A_degenerate)
rank_one_matrix = A_degenerate + split_info["alpha"] * skew(intersection)

xlim, ylim = (-2.2, 2.2), (-1.7, 1.7)
fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.6), gridspec_kw={"width_ratios": [1.25, 1, 1]})
ax = axes[0]
plot_conic(ax, A_degenerate, xlim, ylim, color="#2f5d8c", linewidth=2.2, label="degenerate conic")
plot_line(ax, recovered_g, xlim, ylim, color="#d1495b", linewidth=1.3, linestyle="--", label="recovered line 1")
plot_line(ax, recovered_h, xlim, ylim, color="#edae49", linewidth=1.3, linestyle="--", label="recovered line 2")
scatter_hpoints(ax, [intersection], s=46, color="#222222", zorder=4)
ax.annotate("null point", affine(intersection).real + np.array([0.05, 0.05]), fontsize=8)
ax.set_title("union of two lines")
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal"); ax.grid(True, alpha=0.25); ax.legend(loc="upper right")
for matrix_ax, matrix, title in [(axes[1], A_degenerate.real, "symmetric A"), (axes[2], rank_one_matrix.real, "A + alpha M_p")]:
    norm = TwoSlopeNorm(vcenter=0, vmin=float(np.min(matrix)), vmax=float(np.max(matrix)))
    im = matrix_ax.imshow(matrix, cmap="coolwarm", norm=norm)
    matrix_ax.set_xticks(range(3)); matrix_ax.set_yticks(range(3)); matrix_ax.set_title(title)
    for i in range(3):
        for j in range(3):
            matrix_ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=matrix_ax, fraction=0.046, pad=0.04)
fig.suptitle("Splitting by adding a skew matrix that leaves p.T A p unchanged", y=1.03)
degenerate_split_path = save_figure(fig, "degenerate-conic-splitting.png")

degenerate_checks = {
    "sympy_identity_ok": bool(sympy_identity_ok),
    "split_rank_one_score": split_info["rank_score"],
    "intersection_incidence_original": [float(abs(np.dot(g, intersection))), float(abs(np.dot(h, intersection)))],
    "intersection_incidence_recovered": [float(abs(np.dot(recovered_g, intersection))), float(abs(np.dot(recovered_h, intersection)))],
    "quadratic_form_difference_sample_max": float(max(abs(conic_value(A_degenerate, hpoint(x, y)) - conic_value(rank_one_matrix, hpoint(x, y))) for x, y in [(-1.2, -0.3), (0.4, 0.9), (1.1, -0.8)])),
}
save_json(degenerate_checks, ARTIFACT_ROOT, "checks", "splitting-checks.json")
display_artifact(degenerate_split_path, width=820)

## 2. Why Branches Are Part of the Computation

A rank-1 double line `l l.T` has the same matrix for `l` and `-l`. A closed path of matrices can therefore lift to a vector path whose sign cannot be chosen continuously and single-valuedly everywhere. In code this appears as a pivot search: find a nonzero diagonal, row, column, or minor before extracting coordinates.

In [ ]:
def extract_double_line_with_pivot(A, tol=1e-10):
    diagonal = np.abs(np.diag(A))
    pivot = int(np.argmax(diagonal))
    if diagonal[pivot] < tol:
        raise ValueError("zero double-line matrix")
    return normalize_h(A[:, pivot] / np.sqrt(A[pivot, pivot])), pivot


angles = np.linspace(0, math.pi, 240)
extracted, pivots = [], []
for t in angles:
    line = np.array([math.cos(t), math.sin(t), 0.0])
    v, pivot = extract_double_line_with_pivot(np.outer(line, line))
    extracted.append(v); pivots.append(pivot)
extracted = np.asarray(extracted); pivots = np.asarray(pivots)
jump_sizes = np.linalg.norm(np.diff(extracted[:, :2], axis=0), axis=1)
branching_checks = {
    "endpoint_matrices_equal": bool(np.allclose(np.outer([1, 0, 0], [1, 0, 0]), np.outer([-1, 0, 0], [-1, 0, 0]))),
    "pivot_changes": int(np.sum(np.diff(pivots) != 0)),
    "large_coordinate_jumps": int(np.sum(jump_sizes > 0.25)),
    "max_jump": float(np.max(jump_sizes)),
}

fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.7))
axes[0].plot(np.cos(angles), np.sin(angles), color="#2f5d8c", linewidth=2)
axes[0].scatter([1, -1], [0, 0], color="#d1495b", zorder=3)
axes[0].annotate("same matrix\nfor l and -l", xy=(-1, 0), xytext=(-0.85, 0.35), arrowprops={"arrowstyle": "->", "lw": 0.8}, fontsize=8)
axes[0].set_title("double-line vectors over one matrix loop")
axes[0].set_aspect("equal"); axes[0].grid(True, alpha=0.25)
axes[1].plot(angles, extracted[:, 0].real, label="extracted coord 1", color="#2f5d8c")
axes[1].plot(angles, extracted[:, 1].real, label="extracted coord 2", color="#d1495b")
axes[1].step(angles, pivots / 2.0 - 1.2, where="mid", label="chosen pivot", color="#4d9078", alpha=0.8)
for idx in np.where(jump_sizes > 0.25)[0]:
    axes[1].axvline(angles[idx], color="#222222", linewidth=0.6, alpha=0.35)
axes[1].set_title("pivot extraction is necessarily branched")
axes[1].set_xlabel("t"); axes[1].grid(True, alpha=0.25); axes[1].legend(loc="upper right")
fig.suptitle("Branching appears when a projective line object is forced into vector coordinates", y=1.02)
branching_path = save_figure(fig, "branching-double-line-path.png")
save_json(branching_checks, ARTIFACT_ROOT, "checks", "branching-checks.json")
display_artifact(branching_path, width=820)

## 3. Intersecting a Conic and a Line

For a line `l`, the matrix `M_l.T @ A @ M_l` is a dual conic whose split factors are the two intersection points on `l`. The same recipe covers two real roots, a repeated tangent root, and a complex conjugate pair.

In [ ]:
ellipse = np.diag([0.25, 1.0, -1.0])
line_cases = [
    {"name": "secant", "y": 0.35, "color": "#2f5d8c"},
    {"name": "tangent", "y": 1.00, "color": "#4d9078"},
    {"name": "exterior", "y": 1.25, "color": "#d1495b"},
]
line_rows = []
fig, axes = plt.subplots(1, 3, figsize=(12.3, 3.7), sharex=True, sharey=True)
for ax, item in zip(axes, line_cases):
    y0 = item["y"]
    l = np.array([0.0, 1.0, -y0])
    p, q, info = intersect_conic_line(ellipse, l)
    roots_aff = [affine(p), affine(q)]
    residuals = [abs(conic_value(ellipse, p)), abs(conic_value(ellipse, q)), abs(np.dot(l, p)), abs(np.dot(l, q))]
    discriminant = 4.0 * (1.0 - y0 * y0)
    root_type = "real pair" if discriminant > TOL else "double" if abs(discriminant) <= TOL else "complex conjugate"
    line_rows.append({"case": item["name"], "line_y": y0, "discriminant": discriminant, "root_type": root_type, "split_case": info["case"], "rank_score": info["rank_score"], "max_residual": float(max(abs(r) for r in residuals))})
    plot_conic(ax, ellipse, (-2.5, 2.5), (-1.45, 1.45), color="#222222", linewidth=2.0, label="ellipse")
    plot_line(ax, l, (-2.5, 2.5), (-1.45, 1.45), color=item["color"], linewidth=2.0, linestyle="-", label=f"y = {y0:.2f}")
    if root_type != "complex conjugate":
        scatter_hpoints(ax, [p, q], s=42, color=item["color"], edgecolor="white", linewidth=0.7, zorder=4)
        for r in roots_aff:
            ax.annotate(f"({r[0].real:.2f}, {r[1].real:.2f})", xy=(r[0].real, r[1].real), xytext=(4, 4), textcoords="offset points", fontsize=7)
    else:
        complex_x = roots_aff[0][0]
        ax.scatter([0], [y0], s=48, facecolor="none", edgecolor=item["color"], linewidth=1.5)
        ax.annotate(f"x = +/- {abs(complex_x.imag):.2f} i", xy=(0, y0), xytext=(-46, -30), textcoords="offset points", fontsize=8, arrowprops={"arrowstyle": "->", "lw": 0.7})
    ax.set_title(f"{item['name']}: {root_type}")
    ax.set_aspect("equal"); ax.grid(True, alpha=0.25); ax.legend(loc="lower left", fontsize=7)
axes[0].set_xlim(-2.5, 2.5); axes[0].set_ylim(-1.45, 1.45)
fig.suptitle("Line-conic intersection by splitting a dual degenerate conic", y=1.03)
line_root_path = save_figure(fig, "conic-line-root-splitting.png")
line_summary_path = save_table(line_rows, ARTIFACT_ROOT, "tables", "intersection-summary.csv")
save_json({row["case"]: row for row in line_rows}, ARTIFACT_ROOT, "checks", "line-conic-checks.json")
display_artifact(line_root_path, width=860)
pd.DataFrame(line_rows)

In [ ]:
xs = np.linspace(-2.35, 2.35, 180)
ys = np.linspace(-1.35, 1.35, 180)
X, Y = np.meshgrid(xs, ys)
Z = X * X / 4.0 + Y * Y - 1.0
line_values = np.linspace(-1.25, 1.25, 26)


def line_traces(y0):
    traces = [go.Scatter(x=[-2.4, 2.4], y=[y0, y0], mode="lines", line={"color": "#d1495b", "width": 3}, name="moving line")]
    if abs(y0) <= 1.0:
        xr = 2.0 * math.sqrt(max(0.0, 1.0 - y0 * y0))
        traces.append(go.Scatter(x=[-xr, xr], y=[y0, y0], mode="markers", marker={"size": 10, "color": "#2f5d8c"}, name="real roots"))
    else:
        imag = 2.0 * math.sqrt(y0 * y0 - 1.0)
        traces.append(go.Scatter(x=[0], y=[y0], mode="markers+text", marker={"size": 12, "symbol": "circle-open", "color": "#d1495b"}, text=[f"x=+/-{imag:.2f}i"], textposition="top center", name="complex roots"))
    return traces


base_y = float(line_values[7])
fig = go.Figure(
    data=[go.Contour(x=xs, y=ys, z=Z, contours={"start": 0, "end": 0, "size": 1}, line={"color": "black", "width": 3}, showscale=False, name="ellipse")] + line_traces(base_y),
    frames=[go.Frame(data=line_traces(float(y0)), name=f"{y0:.2f}", traces=[1, 2]) for y0 in line_values],
)
fig.update_layout(
    title="Conic-line root lab: secant, tangent, complex pair",
    width=760,
    height=560,
    xaxis={"range": [-2.5, 2.5], "scaleanchor": "y", "title": "x"},
    yaxis={"range": [-1.4, 1.4], "title": "y"},
    sliders=[{"steps": [{"method": "animate", "label": f"{y0:.2f}", "args": [[f"{y0:.2f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]} for y0 in line_values], "currentvalue": {"prefix": "line y = "}}],
    updatemenus=[{"type": "buttons", "buttons": [{"label": "play", "method": "animate", "args": [None, {"frame": {"duration": 180, "redraw": True}, "fromcurrent": True}]}, {"label": "pause", "method": "animate", "args": [[None], {"frame": {"duration": 0}, "mode": "immediate"}]}]}],
)
interactive_line_path = HTML / "conic-line-root-discriminant-lab.html"
fig.write_html(interactive_line_path, include_plotlyjs=True, include_mathjax=False, full_html=True)
display_artifact(interactive_line_path, width="100%", height=560)

## 4. Intersecting Two Conics

A pencil member `t A + B` is degenerate exactly when `det(t A + B) = 0`. In this example the conics are built through four common points, so the three real roots of the determinant cubic correspond to the three pairings of those four points by line pairs. Choosing one root reduces conic-conic intersection to two conic-line intersections.

In [ ]:
common_points = [hpoint(-1.25, -0.55), hpoint(-0.35, 0.92), hpoint(0.65, -0.82), hpoint(1.30, 0.45)]
A_conic = conic_from_five(common_points + [hpoint(0.0, -1.40)])
B_conic = conic_from_five(common_points + [hpoint(0.05, 1.35)])
coeffs = pencil_cubic_coefficients(A_conic, B_conic)
roots = np.array(sorted(np.roots(coeffs), key=lambda z: z.real))
selected_root = roots[1]
selected_degenerate = selected_root * A_conic + B_conic
line_a, line_b, pencil_split_info = split_degenerate_conic(selected_degenerate)
intersections = []
for line in [line_a, line_b]:
    p, q, _ = intersect_conic_line(A_conic, line)
    intersections.extend([p, q])

pencil_rows = []
for root in roots:
    C = root * A_conic + B_conic
    u, v, info = split_degenerate_conic(C)
    pencil_rows.append({"root_t": float(root.real) if abs(root.imag) < 1e-9 else str(root), "det_residual": float(abs(np.linalg.det(C))), "rank_score_after_split": info["rank_score"], "line_pair_real": bool(abs(root.imag) < 1e-8 and np.linalg.norm(np.asarray(u).imag) < 1e-7 and np.linalg.norm(np.asarray(v).imag) < 1e-7)})
common_residuals = [max(abs(conic_value(A_conic, p)), abs(conic_value(B_conic, p))) for p in intersections]

fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.4))
xlim, ylim = (-1.8, 1.7), (-1.55, 1.55)
ax = axes[0, 0]
plot_conic(ax, A_conic, xlim, ylim, color="#2f5d8c", linewidth=2.1, label="conic A")
plot_conic(ax, B_conic, xlim, ylim, color="#d1495b", linewidth=2.1, linestyle="--", label="conic B")
scatter_hpoints(ax, common_points, s=40, color="#222222", zorder=4)
for idx, p in enumerate(common_points, start=1):
    ax.annotate(str(idx), xy=affine(p).real, xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("two conics through four shared points")
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal"); ax.grid(True, alpha=0.25); ax.legend(loc="lower right")

ax = axes[0, 1]
tgrid = np.linspace(float(roots[0].real) - 0.2, float(roots[-1].real) + 0.2, 400)
ax.plot(tgrid, [np.linalg.det(t * A_conic + B_conic).real for t in tgrid], color="#222222")
ax.axhline(0, color="#888888", linewidth=0.8)
ax.scatter(roots.real, [0] * len(roots), color="#d1495b", zorder=4)
for root in roots:
    ax.annotate(f"{root.real:.3f}", xy=(root.real, 0), xytext=(-12, 8), textcoords="offset points", fontsize=8)
ax.set_title("det(t A + B) has three degenerate roots")
ax.set_xlabel("t = lambda / mu"); ax.set_ylabel("determinant"); ax.grid(True, alpha=0.25)

ax = axes[1, 0]
for root, color in zip(roots, ["#4d9078", "#edae49", "#b56576"]):
    u, v, _ = split_degenerate_conic(root * A_conic + B_conic)
    plot_line(ax, u, xlim, ylim, color=color, linewidth=1.4, linestyle="-")
    plot_line(ax, v, xlim, ylim, color=color, linewidth=1.4, linestyle="--")
scatter_hpoints(ax, common_points, s=38, color="#222222", zorder=4)
ax.set_title("the three degenerate pencil members")
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal"); ax.grid(True, alpha=0.25)

ax = axes[1, 1]
plot_conic(ax, A_conic, xlim, ylim, color="#2f5d8c", linewidth=1.9, label="conic A")
plot_line(ax, line_a, xlim, ylim, color="#d1495b", linewidth=2.0, linestyle="-", label="split line 1")
plot_line(ax, line_b, xlim, ylim, color="#edae49", linewidth=2.0, linestyle="-", label="split line 2")
scatter_hpoints(ax, intersections, s=44, color="#222222", zorder=4)
ax.set_title("one root reduces the problem to two line solves")
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal"); ax.grid(True, alpha=0.25); ax.legend(loc="lower right", fontsize=7)
fig.suptitle("Conic-conic intersection through the determinant cubic of the pencil", y=0.995)
fig.tight_layout()
pencil_path = save_figure(fig, "conic-pencil-degenerate-members.png")
pencil_table_path = save_table(pencil_rows, ARTIFACT_ROOT, "tables", "pencil-cubic-summary.csv")
pencil_checks = {
    "cubic_coefficients": [complex(c).__repr__() for c in coeffs],
    "roots": [complex(r).__repr__() for r in roots],
    "max_det_residual": float(max(row["det_residual"] for row in pencil_rows)),
    "selected_split_rank_score": pencil_split_info["rank_score"],
    "max_common_point_residual": float(max(abs(r) for r in common_residuals)),
    "intersection_count_from_selected_member": len(intersections),
}
save_json(pencil_checks, ARTIFACT_ROOT, "checks", "pencil-cubic-checks.json")
display_artifact(pencil_path, width=860)
pd.DataFrame(pencil_rows)

## 5. Complex Roots Can Still Encode Real Geometry

When a real line misses a real conic, the algebraic intersections are a conjugate complex pair. Projectively, that is not a failure: the join of conjugate complex points is a real line after removing one common complex scalar.

In [ ]:
outside_line = np.array([0.0, 1.0, -1.25])
complex_p, complex_q, complex_info = intersect_conic_line(ellipse, outside_line)
real_join = real_representative(join_points(complex_p, complex_q))
join_residual = float(np.linalg.norm(normalize_h(real_join) - normalize_h(outside_line)))
complex_roots = [affine(complex_p)[0], affine(complex_q)[0]]

fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.9))
ax = axes[0]
plot_conic(ax, ellipse, (-2.5, 2.5), (-1.45, 1.45), color="#222222", linewidth=2.0)
plot_line(ax, outside_line, (-2.5, 2.5), (-1.45, 1.45), color="#d1495b", linewidth=2.0)
ax.scatter([0], [1.25], facecolor="none", edgecolor="#d1495b", s=62, linewidth=1.4)
ax.annotate("no real point here", xy=(0, 1.25), xytext=(-60, -32), textcoords="offset points", arrowprops={"arrowstyle": "->", "lw": 0.8}, fontsize=8)
ax.set_title("real view: exterior line")
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-1.45, 1.45); ax.set_aspect("equal"); ax.grid(True, alpha=0.25)

ax = axes[1]
ax.axhline(0, color="#777777", linewidth=0.8); ax.axvline(0, color="#777777", linewidth=0.8)
ax.scatter([z.real for z in complex_roots], [z.imag for z in complex_roots], s=70, color=["#2f5d8c", "#d1495b"])
for z, label in zip(complex_roots, ["p", "conj(p)"]):
    ax.annotate(label, xy=(z.real, z.imag), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.plot([complex_roots[0].real, complex_roots[1].real], [complex_roots[0].imag, complex_roots[1].imag], color="#4d9078", linewidth=1.4)
ax.text(0.04, 0.05, "join(p, conj(p)) rescales to the original real line", transform=ax.transAxes, fontsize=8)
ax.set_title("root coordinate x in the complex plane")
ax.set_xlabel("real x"); ax.set_ylabel("imaginary x"); ax.set_aspect("equal"); ax.grid(True, alpha=0.25)
fig.suptitle("Complex conjugate intersections carry a real projective line", y=1.02)
complex_path = save_figure(fig, "complex-conjugate-reality-check.png")
complex_checks = {
    "complex_split_rank_score": complex_info["rank_score"],
    "join_residual_to_original_line": join_residual,
    "root_imaginary_parts": [float(z.imag) for z in complex_roots],
    "line_incidence_residuals": [float(abs(np.dot(outside_line, complex_p))), float(abs(np.dot(outside_line, complex_q)))],
    "conic_residuals": [float(abs(conic_value(ellipse, complex_p))), float(abs(conic_value(ellipse, complex_q)))],
}
save_json(complex_checks, ARTIFACT_ROOT, "checks", "complex-root-checks.json")
display_artifact(complex_path, width=820)

## 6. Four Points and One Tangent Line

The final construction asks for conics through four fixed points and tangent to a line. The computational reduction is to find the two fixed points of the involution induced on the line by the conic bundle through the four points. Once those two tangent points are known, each desired conic is just a five-point conic.

In [ ]:
def tangent_points_from_four_points(A, B, C, D, l, o):
    a1 = meet_lines(join_points(A, C), l)
    a2 = meet_lines(join_points(B, D), l)
    b1 = meet_lines(join_points(A, B), l)
    b2 = meet_lines(join_points(C, D), l)
    first = bracket(o, a2, b1) * bracket(o, a2, b2)
    second = bracket(o, a1, b1) * bracket(o, a1, b2)
    u, v = np.sqrt(complex(first)), np.sqrt(complex(second))
    return {"a1": a1, "a2": a2, "b1": b1, "b2": b2, "x": real_representative(u * a1 + v * a2), "y": real_representative(u * a1 - v * a2), "products": (first, second)}


TA, TB, TC, TD = hpoint(0.523, -0.087), hpoint(1.064, 0.035), hpoint(-0.245, 0.749), hpoint(0.002, 1.004)
tangent_line = np.array([0.0, 1.0, 0.8])
off_line_point = hpoint(0.0, 0.2)
tangent_data = tangent_points_from_four_points(TA, TB, TC, TD, tangent_line, off_line_point)
conic_x = conic_from_five([TA, TB, TC, TD, tangent_data["x"]])
conic_y = conic_from_five([TA, TB, TC, TD, tangent_data["y"]])
tangent_residual_x = float(abs(tangent_line @ adjugate(conic_x) @ tangent_line))
tangent_residual_y = float(abs(tangent_line @ adjugate(conic_y) @ tangent_line))
point_residuals_x = [float(abs(conic_value(conic_x, p))) for p in [TA, TB, TC, TD, tangent_data["x"]]]
point_residuals_y = [float(abs(conic_value(conic_y, p))) for p in [TA, TB, TC, TD, tangent_data["y"]]]

fig, ax = plt.subplots(figsize=(8.7, 4.9))
xlim, ylim = (-3.95, 2.35), (-1.25, 1.35)
plot_conic(ax, conic_x, xlim, ylim, color="#2f5d8c", linewidth=2.0, label="tangent conic through x")
plot_conic(ax, conic_y, xlim, ylim, color="#d1495b", linewidth=2.0, linestyle="--", label="tangent conic through y")
plot_line(ax, tangent_line, xlim, ylim, color="#222222", linewidth=1.6, linestyle="-", label="given tangent line")
scatter_hpoints(ax, [TA, TB, TC, TD], s=46, color="#222222", zorder=5)
for label, point in zip(["A", "B", "C", "D"], [TA, TB, TC, TD]):
    ax.annotate(label, xy=affine(point).real, xytext=(5, 4), textcoords="offset points", fontsize=9)
for label, point, color in [("x", tangent_data["x"], "#2f5d8c"), ("y", tangent_data["y"], "#d1495b")]:
    xy = affine(point).real
    ax.scatter([xy[0]], [xy[1]], s=58, color=color, edgecolor="white", linewidth=0.8, zorder=6)
    ax.annotate(f"tangent {label}", xy=xy, xytext=(6, -14), textcoords="offset points", fontsize=8, color=color)
for label in ["a1", "a2", "b1", "b2"]:
    xy = affine(tangent_data[label]).real
    ax.scatter([xy[0]], [xy[1]], s=25, color="#edae49", zorder=4)
    ax.annotate(label, xy=xy, xytext=(3, 5), textcoords="offset points", fontsize=7)
ax.set_title("Two conics through four points tangent to one line")
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal"); ax.grid(True, alpha=0.25); ax.legend(loc="upper left", fontsize=8)
tangent_path = save_figure(fig, "tangent-conics-four-points-line.png")
tangent_checks = {
    "bracket_products": [complex(v).__repr__() for v in tangent_data["products"]],
    "tangent_residuals": [tangent_residual_x, tangent_residual_y],
    "max_point_residual_x": float(max(point_residuals_x)),
    "max_point_residual_y": float(max(point_residuals_y)),
    "tangent_points_on_line": [float(abs(np.dot(tangent_line, tangent_data["x"]))), float(abs(np.dot(tangent_line, tangent_data["y"])))],
}
save_json(tangent_checks, ARTIFACT_ROOT, "checks", "tangent-constraint-checks.json")
display_artifact(tangent_path, width=820)
tangent_checks

## 7. Decision-Tree Checks

Chapter 11 repeatedly asks the implementer to inspect a matrix or polynomial before choosing a formula. The graph and table turn those branch points into explicit checks.

In [ ]:
decision_rows = [
    {"operation": "split degenerate conic", "branch trigger": "rank(A) is 1 or 2", "action": "double line: choose nonzero row or column; pair: find null point and skew alpha", "invariant": "rank-one score after skew correction"},
    {"operation": "choose split pivot", "branch trigger": "largest nonzero entry of rank-one matrix", "action": "use its column and row as projective factors", "invariant": "factors vanish on sampled points of the union"},
    {"operation": "line-conic", "branch trigger": "dual pair is real pair, double, or complex pair", "action": "split M_l.T A M_l", "invariant": "both roots satisfy l dot p = 0 and p.T A p = 0"},
    {"operation": "conic-conic", "branch trigger": "roots of det(lambda A + mu B)", "action": "select a degenerate pencil member", "invariant": "det residual near zero and four common points recovered"},
    {"operation": "complex reality", "branch trigger": "root coordinates have imaginary parts", "action": "test for a common complex scalar", "invariant": "rescaled representative is real"},
    {"operation": "four points plus tangent", "branch trigger": "two fixed points of the induced involution", "action": "build two five-point conics", "invariant": "l.T adj(C) l = 0"},
]
decision_table_path = save_table(decision_rows, ARTIFACT_ROOT, "tables", "decision-tree-checks.csv")
G = nx.DiGraph()
G.add_edges_from([
    ("input conic matrix", "det/rank test"), ("det/rank test", "rank 2 line pair"), ("det/rank test", "rank 1 double line"),
    ("rank 2 line pair", "null point + skew alpha"), ("rank 1 double line", "nonzero row/column pivot"),
    ("null point + skew alpha", "rank-one factors"), ("nonzero row/column pivot", "rank-one factors"),
    ("input line and conic", "dual pair M_l.T A M_l"), ("dual pair M_l.T A M_l", "rank-one factors"),
    ("two conics", "pencil determinant cubic"), ("pencil determinant cubic", "degenerate pencil member"),
    ("degenerate pencil member", "rank 2 line pair"), ("rank-one factors", "residual checks"),
    ("residual checks", "real / tangent / complex classification"),
    ("four points + line", "involution fixed points"), ("involution fixed points", "two tangent conics"), ("two tangent conics", "residual checks"),
])
pos = {
    "input conic matrix": (0, 3), "det/rank test": (1.6, 3), "rank 2 line pair": (3.2, 3.5), "rank 1 double line": (3.2, 2.5),
    "null point + skew alpha": (4.9, 3.5), "nonzero row/column pivot": (4.9, 2.5), "input line and conic": (0, 1.45),
    "dual pair M_l.T A M_l": (2.6, 1.45), "two conics": (0, 0.2), "pencil determinant cubic": (2.25, 0.2),
    "degenerate pencil member": (4.5, 0.2), "rank-one factors": (6.8, 2.55), "residual checks": (8.8, 2.0),
    "real / tangent / complex classification": (10.9, 2.0), "four points + line": (0, -1.05), "involution fixed points": (2.65, -1.05),
    "two tangent conics": (5.0, -1.05),
}
fig, ax = plt.subplots(figsize=(13.2, 5.2))
node_colors = ["#f1f5f9" if "input" in node or node in {"two conics", "four points + line"} else "#fff7e6" if "test" in node or "cubic" in node or "pivot" in node else "#e8f3ee" for node in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", width=1.0, edge_color="#555555", connectionstyle="arc3,rad=0.03")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color=node_colors, edgecolors="#333333", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7.4)
ax.set_title("Decision tree for Chapter 11 conic computations")
ax.axis("off")
decision_path = save_figure(fig, "decision-tree-checks.png")
save_json({"rows": len(decision_rows), "nodes": G.number_of_nodes(), "edges": G.number_of_edges()}, ARTIFACT_ROOT, "checks", "decision-tree-checks.json")
display_artifact(decision_path, width=920)
pd.DataFrame(decision_rows)

## Applied Lab

Use the Plotly line-conic lab as a controlled experiment. Move the horizontal line from inside the ellipse to outside it and watch which entry in the decision tree changes first: the conic matrix is unchanged, the line matrix has the same form, but the split of `M_l.T @ A @ M_l` changes from two real points to one double point and then to a conjugate complex pair.

Then compare that behavior with the conic pencil figure. The determinant cubic does not directly return the four intersection points; it returns line-pair conics inside the pencil. The actual points appear only after the selected degenerate member is split and each line is intersected with one original conic. This is the chapter's main design pattern: create a degenerate object whose split reduces the hard problem to a previously solved primitive.

## Artifact Gallery

The static artifacts are linked directly so the notebook remains inspectable in rendered form after execution.

![Degenerate conic splitting](../../artifacts/chapter-11-calculating-with-conics/figures/degenerate-conic-splitting.png)

![Conic-line root splitting](../../artifacts/chapter-11-calculating-with-conics/figures/conic-line-root-splitting.png)

![Conic pencil degenerate members](../../artifacts/chapter-11-calculating-with-conics/figures/conic-pencil-degenerate-members.png)

![Tangent conics through four points](../../artifacts/chapter-11-calculating-with-conics/figures/tangent-conics-four-points-line.png)

## Final Sanity Checks

The final cell checks the algebraic identities, root residuals, artifact sizes, and JSON summary data. A chapter notebook is not complete until these invariants pass in a fresh execution.

In [ ]:
visual_artifacts = [degenerate_split_path, branching_path, line_root_path, interactive_line_path, pencil_path, complex_path, tangent_path, decision_path]
table_artifacts = [line_summary_path, pencil_table_path, decision_table_path]
check_artifacts = [
    storyboard_path,
    CHECKS / "splitting-checks.json",
    CHECKS / "branching-checks.json",
    CHECKS / "line-conic-checks.json",
    CHECKS / "pencil-cubic-checks.json",
    CHECKS / "complex-root-checks.json",
    CHECKS / "tangent-constraint-checks.json",
    CHECKS / "decision-tree-checks.json",
]
assert_artifacts(visual_artifacts, min_size=256)
assert_artifacts(table_artifacts + check_artifacts, min_size=32)

sample = [-1.4, -0.2, 0.75, 1.6]
image = [(1.1 * x - 0.25) / (0.22 * x + 1.0) for x in sample]
cross_ratio_error = abs(cross_ratio(*sample) - cross_ratio(*image))
assert sympy_identity_ok
assert degenerate_checks["split_rank_one_score"] < 1e-8
assert degenerate_checks["quadratic_form_difference_sample_max"] < 1e-8
assert branching_checks["endpoint_matrices_equal"]
assert max(row["max_residual"] for row in line_rows) < 1e-7
assert pencil_checks["max_det_residual"] < 1e-10
assert pencil_checks["max_common_point_residual"] < 1e-7
assert complex_checks["join_residual_to_original_line"] < 1e-7
assert max(complex_checks["conic_residuals"]) < 1e-7
assert max(tangent_checks["tangent_residuals"]) < 1e-7
assert max(tangent_checks["max_point_residual_x"], tangent_checks["max_point_residual_y"]) < 1e-7
assert cross_ratio_error < 1e-9

stats = {}
for path in visual_artifacts:
    if path.suffix.lower() == ".png":
        item = image_stats(path)
        item["path"] = book_relative(path)
    else:
        item = {"file_size": path.stat().st_size}
    stats[book_relative(path)] = item
visual_checks = {
    "chapter": 11,
    "all_files_exist": all(path.exists() for path in visual_artifacts + table_artifacts + check_artifacts),
    "cross_ratio_error": float(cross_ratio_error),
    "numeric_checks": {
        "split_rank_one_score": degenerate_checks["split_rank_one_score"],
        "line_conic_max_residual": float(max(row["max_residual"] for row in line_rows)),
        "pencil_max_det_residual": pencil_checks["max_det_residual"],
        "pencil_max_common_point_residual": pencil_checks["max_common_point_residual"],
        "complex_join_residual": complex_checks["join_residual_to_original_line"],
        "tangent_max_residual": float(max(tangent_checks["tangent_residuals"])),
    },
    "artifacts": sorted([book_relative(path) for path in visual_artifacts + table_artifacts]),
    "image_stats": stats,
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
final = {
    "chapter": 11,
    "source_span": "printed pp. 189-208 / PDF pp. 211-230",
    "checks_passed": True,
    "core_identities": {"skew_split_identity": bool(sympy_identity_ok), "cross_ratio_error": float(cross_ratio_error)},
    "artifact_count": len(visual_artifacts) + len(table_artifacts),
    "artifacts": sorted([book_relative(path) for path in visual_artifacts + table_artifacts + check_artifacts + [visual_checks_path]]),
    "numeric_checks": visual_checks["numeric_checks"],
}
final_path = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
print(json.dumps(final, indent=2))

## Takeaways

- A conic matrix is an executable object: evaluation, polarity, tangency, degeneracy, and intersection all become matrix operations.
- Degenerate conics are the bridge. Splitting a line pair turns a hard intersection problem into simpler line operations.
- Branches are not an implementation flaw. The algebraic object may be continuous while any coordinate representative needs pivot choices, root choices, and rank tests.
- Complex roots are part of real projective computation. Conjugate complex intersections can still determine real lines and real degenerate pencil members.
- The tangent-through-four-points construction is another reduction: find two fixed points on the line, then build two ordinary five-point conics and validate tangency through the dual conic condition.